# Sesión de clase 06 · Ejercicio 1: búsqueda semántica en `comentarios.csv`

Hasta la sesión anterior sabíamos convertir un texto en un vector y comparar
dos vectores con similitud coseno. Hoy resolvemos el siguiente problema:
dada una consulta, encontrar los k textos más parecidos entre **miles o
millones** de comentarios, sin depender de que compartan las mismas
palabras.

Este notebook reproduce el **Ejercicio 1** de la clase: comparar una
búsqueda con `WHERE texto LIKE '%lento%'` contra una búsqueda semántica con
**ChromaDB**, sobre los 135 comentarios reales de la tienda del curso.

**Datos:** `data/comentarios.csv` — `cliente_id`, `cliente_ciudad`,
`cliente_pais`, `calificacion`, `fecha`, `texto`.

**Modelo de embeddings:** `paraphrase-multilingual-MiniLM-L12-v2` (384
dimensiones, multilingüe, funciona en CPU).

In [1]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

pd.set_option("display.max_colwidth", 100)

## 1. Cargar

Leemos `comentarios.csv` con pandas. Nos interesan sobre todo `texto`,
`calificacion` y `cliente_ciudad`.

In [2]:
comentarios = pd.read_csv("../data/comentarios.csv")
print(comentarios.shape)
comentarios[["cliente_id", "cliente_ciudad", "cliente_pais", "calificacion", "texto"]].head()

(135, 8)


,cliente_id,cliente_ciudad,cliente_pais,calificacion,texto
0,35,Trujillo,Peru,5,El proceso de facturacion para empresa fue mas simple de lo que esperaba.
1,90,Cuenca,Ecuador,3,"En general cumple lo que promete, ni mejor ni peor que otras tiendas de tecnologia que he probado."
2,31,Arequipa,Peru,4,"Buena tienda en general, seguire comprando aqui mis proximos equipos."
3,27,Lima,Peru,3,El envio internacional tardo mas dias de los indicados originalmente.
4,84,Rosario,Argentina,3,El proceso de devolucion fue sencillo de iniciar aunque tomo unos dias en confirmarse.


## ¿Por qué no alcanza con `LIKE`?

Necesidad: encontrar quejas sobre la lentitud del sitio web. La consulta SQL
equivalente en pandas es `texto.str.contains("lento")`.

In [3]:
like_lento = comentarios[comentarios["texto"].str.contains("lento", case=False)]
print(f"{len(like_lento)} resultado(s) con LIKE '%lento%'")
like_lento[["calificacion", "texto"]]

3 resultado(s) con LIKE '%lento%'


,calificacion,texto
15,2,El proceso de cambio de producto por otro modelo fue mas lento de lo esperado.
41,3,"El soporte responde en tiempos razonables, ni muy rapido ni muy lento comparado con otras tiendas."
117,1,La verificacion de identidad para mi primera compra tomo mas de tres dias habiles. Fue un proces...


Los comentarios que sí hablan de un sitio lento no siempre usan la palabra
«lento»: dicen «tardan en cargar» o «podría cargar más rápido». `LIKE` no
los encuentra porque busca la cadena exacta, no el significado. Y si
ampliamos el patrón para intentar capturar variantes, aparecen falsos
positivos:

In [4]:
like_lent = comentarios[comentarios["texto"].str.contains("lent", case=False)]
print(f"{len(like_lent)} resultado(s) con LIKE '%lent%'")
like_lent[["calificacion", "texto"]]

8 resultado(s) con LIKE '%lent%'


,calificacion,texto
15,2,El proceso de cambio de producto por otro modelo fue mas lento de lo esperado.
23,5,Excelente que muestren opiniones verificadas de compradores reales.
41,3,"El soporte responde en tiempos razonables, ni muy rapido ni muy lento comparado con otras tiendas."
43,5,El servicio postventa fue excelente cuando necesite activar la garantia de mi laptop.
110,5,"Excelente experiencia de principio a fin, desde la busqueda hasta la entrega del producto. Defin..."
114,2,La pagina se puso lenta durante una oferta de temporada y perdi el producto que queria comprar. ...
117,1,La verificacion de identidad para mi primera compra tomo mas de tres dias habiles. Fue un proces...
123,5,"Excelente variedad de marcas, encontre opciones que no tenia en otras tiendas."


`%lent%` no solo trae los comentarios sobre lentitud: también atrapa
«Excelente» porque la subcadena «lent» aparece ahí por coincidencia. Es el
problema opuesto: ahora hay ruido. Esto es lo que motiva la búsqueda
semántica.

## 2. Vectorizar y construir la colección de Chroma

Tres pasos: crear la colección (eligiendo el modelo de embedding y la
métrica), agregar documentos (Chroma los vectoriza automáticamente) y
consultar (la consulta se vectoriza con el mismo modelo).

Usamos `Client()` en memoria: alcanza para este ejercicio y se pierde al
cerrar el proceso. La persistencia en disco (`PersistentClient`) se usa en
el laboratorio.

In [5]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

cliente = chromadb.Client()
col_comentarios = cliente.get_or_create_collection(
    name="comentarios",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

col_comentarios.add(
    ids=comentarios["id"].astype(str).tolist(),
    documents=comentarios["texto"].tolist(),
    metadatas=comentarios[["cliente_ciudad", "cliente_pais", "calificacion"]].to_dict("records"),
)

print(col_comentarios.count(), "comentarios indexados")

/Users/erichuiza/Documents/pucp/miería web/2026-2/dev/sesion-de-clase-06/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7215.37it/s]


135 comentarios indexados


## 3. Consultar

`query()` devuelve **distancias**, no similitudes: con métrica coseno,
`similitud = 1 - distancia`.

In [6]:
def buscar(coleccion, consulta, n_results=5):
    res = coleccion.query(query_texts=[consulta], n_results=n_results)
    filas = []
    for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
        filas.append({
            "similitud": round(1 - dist, 3),
            "ciudad": meta.get("cliente_ciudad"),
            "calificacion": meta.get("calificacion"),
            "texto": doc,
        })
    return pd.DataFrame(filas)

buscar(col_comentarios, "el sitio web es lento")

,similitud,ciudad,calificacion,texto
0,0.651,Chimbote,2,La pagina se puso lenta durante una oferta de temporada y perdi el producto que queria comprar. ...
1,0.548,Cusco,3,El sitio funciona bien en general aunque algunas paginas tardan un poco mas de lo esperado en ca...
2,0.446,Trujillo,2,El sitio no guarda bien las preferencias de idioma y cada vez que entro tengo que cambiarlo de n...
3,0.437,Montevideo,3,El diseno del sitio en modo oscuro tiene algunos textos dificiles de leer.
4,0.414,Chiclayo,3,El diseno del sitio es agradable pero en modo oscuro algunos textos son dificiles de leer. Peque...


Comparado con `LIKE '%lento%'` (3 resultados, ninguno relevante) y
`LIKE '%lent%'` (8 resultados, la mitad ruido), la búsqueda semántica
recupera los comentarios sobre lentitud del sitio aunque usen palabras
distintas («tardan en cargar», «podría cargar más rápido»).

In [7]:
buscar(col_comentarios, "tuve problemas para pagar con mi tarjeta")

,similitud,ciudad,calificacion,texto
0,0.647,Trujillo,2,Tuve problemas para aplicar un cupon de descuento durante el checkout.
1,0.621,Cusco,2,La factura electronica llego con datos incorrectos y tuve que pedir que la corrigieran.
2,0.614,Ciudad de Mexico,1,Me cobraron dos veces por error y tuve que esperar varios dias para que me reembolsaran.
3,0.602,Arequipa,4,"El proceso de pago con tarjeta fue rapido, solo me hubiera gustado mas metodos de pago."
4,0.580,Chiclayo,1,Cancele un pedido y el reembolso tardo mas de diez dias habiles en aparecer en mi tarjeta. El so...


In [8]:
buscar(col_comentarios, "quiero devolver un producto")

,similitud,ciudad,calificacion,texto
0,0.537,Santiago,3,"El sitio se cayo un momento durante una oferta especial, tuve que reintentar la compra."
1,0.515,Manta,5,El programa de puntos por compras es un buen incentivo para volver a comprar.
2,0.511,Santiago,4,Muy buena atencion cuando pregunte por la disponibilidad de un modelo agotado. Me avisaron apena...
3,0.493,Medellin,1,Compre un producto marcado como disponible y luego me avisaron que no habia stock.
4,0.483,Chimbote,2,El precio de un producto cambio de un dia para otro sin ninguna explicacion clara. Senti que per...


## 4. Umbral de similitud

`query()` siempre devuelve `n_results` documentos, aunque ninguno tenga
relación con la consulta. El umbral es la similitud mínima para aceptar un
resultado; con embeddings multilingües suele estar entre 0.35 y 0.50, y se
ajusta probando con datos propios.

In [9]:
UMBRAL = 0.45

def buscar_con_umbral(coleccion, consulta, umbral=UMBRAL, n_results=10):
    df = buscar(coleccion, consulta, n_results=n_results)
    return df[df["similitud"] >= umbral]

print("Resultados que superan el umbral (0.45):")
buscar_con_umbral(col_comentarios, "el sitio web es lento")

Resultados que superan el umbral (0.45):


,similitud,ciudad,calificacion,texto
0,0.651,Chimbote,2,La pagina se puso lenta durante una oferta de temporada y perdi el producto que queria comprar. ...
1,0.548,Cusco,3,El sitio funciona bien en general aunque algunas paginas tardan un poco mas de lo esperado en ca...


Con un umbral más bajo entran más resultados, algunos menos relacionados;
con uno más alto se descartan resultados dudosos pero también algunos
relevantes. No hay un valor «correcto» - se fija probando consultas reales
y viendo dónde empieza a aparecer ruido.

## Discusión: similitud semántica no es lo mismo que sentimiento

La consulta sobre pagos con tarjeta trae comentarios sobre el **tema**
pago/tarjeta, sin importar si la experiencia fue buena o mala.

In [10]:
resultado_pago = buscar(col_comentarios, "tuve problemas para pagar con mi tarjeta", n_results=5)
resultado_pago

,similitud,ciudad,calificacion,texto
0,0.647,Trujillo,2,Tuve problemas para aplicar un cupon de descuento durante el checkout.
1,0.621,Cusco,2,La factura electronica llego con datos incorrectos y tuve que pedir que la corrigieran.
2,0.614,Ciudad de Mexico,1,Me cobraron dos veces por error y tuve que esperar varios dias para que me reembolsaran.
3,0.602,Arequipa,4,"El proceso de pago con tarjeta fue rapido, solo me hubiera gustado mas metodos de pago."
4,0.580,Chiclayo,1,Cancele un pedido y el reembolso tardo mas de diez dias habiles en aparecer en mi tarjeta. El so...


**Preguntas de discusión**

- ¿Por qué puede aparecer en el top-5 un comentario con calificación alta
  (5★) sobre pagos, aunque la consulta hable de un *problema*? (La
  similitud semántica mide de qué tema habla el texto, no si la experiencia
  fue positiva o negativa.)
- ¿Qué umbral usarían para descartar resultados poco relacionados? Prueben
  distintos valores en `buscar_con_umbral` y observen dónde el resultado
  deja de tener sentido.
- ¿Qué pasaría si compararan la consulta contra `resenas_entrega.csv` en
  vez de `comentarios.csv`? (Se explora en el laboratorio.)